# 04. 프리빌트 미들웨어 (Prebuilt Middleware)

> 직접 만들기 전에 **이미 있는 12+ 개 프리빌트 미들웨어**부터 알아두세요. Summarization · PII · CallLimit · Fallback 등 자주 쓰는 도구를 한 곳에 모았어요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. LangChain V1이 제공하는 12개+ 프리빌트 미들웨어의 역할과 용도를 구별할 수 있어요
2. `SummarizationMiddleware`의 트리거 조건(tokens/messages/fraction)을 상황에 맞게 설정할 수 있어요
3. `PIIMiddleware`로 이메일, 신용카드, IP 등 4가지 PII 유형을 4가지 전략(redact/mask/hash/block)으로 처리할 수 있어요
4. `ModelFallbackMiddleware`, `ToolRetryMiddleware`, `ModelCallLimitMiddleware` 등 복원력(Resilience) 미들웨어를 조합하여 안정적인 에이전트를 만들 수 있어요
5. `LLMToolSelectorMiddleware`, `TodoListMiddleware` 등 에이전트 성능 향상 미들웨어를 적용할 수 있어요

## 사전 지식

- `03-Context-Engineering.ipynb`: Model/Tool/Life-cycle Context + Runtime/State/Store 데이터 소스
- `01-Middleware-Basics.ipynb`: `AgentMiddleware` 클래스와 훅 종류 (`before_model`, `wrap_tool_call` 등)

## 프리빌트 미들웨어란?

직접 구현하지 않아도 LangChain V1이 즉시 사용 가능한 미들웨어를 12가지 이상 제공해요. 이 미들웨어들은 실무에서 자주 필요한 패턴을 정교하게 구현해 놓은 것이에요.

> **왜 프리빌트 미들웨어를 사용할까요?** 직접 구현하면 수십 줄의 코드가 필요한 기능을 한 줄로 추가할 수 있어요. 마치 스마트폰에 앱을 설치하는 것처럼, `middleware=[PIIMiddleware("email", strategy="redact")]` 한 줄이면 이메일 보호 기능이 추가돼요. 직접 정규식을 짜고 테스트하는 수고를 덜 수 있어요.

```mermaid
flowchart TD
    subgraph Memory["대화 관리"]
        SUM["SummarizationMiddleware<br/>토큰/메시지/비율 기반 요약"]
        CTX["ContextEditingMiddleware<br/>도구 사용 기록 정리"]
    end

    subgraph Privacy["보안 / 프라이버시"]
        PII["PIIMiddleware<br/>redact / mask / hash / block"]
    end

    subgraph Control["흐름 제어"]
        HITL["HumanInTheLoopMiddleware<br/>approve / edit / reject"]
        MCL["ModelCallLimitMiddleware<br/>thread / run 횟수 제한"]
        TCL["ToolCallLimitMiddleware<br/>전역 / 도구별 제한"]
    end

    subgraph Resilience["복원력"]
        MFB["ModelFallbackMiddleware<br/>캐스케이딩 폴백"]
        TRT["ToolRetryMiddleware<br/>지수 백오프 재시도"]
        MRT["ModelRetryMiddleware<br/>모델 재시도"]
    end

    subgraph Performance["성능 향상"]
        LTS["LLMToolSelectorMiddleware<br/>필요한 도구만 선택"]
        TDL["TodoListMiddleware<br/>write_todos 도구 주입"]
        LTE["LLMToolEmulatorMiddleware<br/>실행 없이 도구 응답 생성"]
    end

    classDef memory fill:#d4edda,stroke:#28a745,color:#155724
    classDef privacy fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef control fill:#cce5ff,stroke:#007bff,color:#004085
    classDef resilience fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef performance fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class SUM,CTX memory
    class PII privacy
    class HITL,MCL,TCL control
    class MFB,TRT,MRT resilience
    class LTS,TDL,LTE performance
```

### 프리빌트 미들웨어 전체 목록

| 카테고리 | 미들웨어 | 핵심 용도 |
|---------|---------|----------|
| **대화 관리** | `SummarizationMiddleware` | 토큰 한도 초과 전 자동 요약 |
| **대화 관리** | `ContextEditingMiddleware` | 도구 사용 기록(ToolUse) 제거 |
| **보안** | `PIIMiddleware` | 개인식별정보 감지 및 처리 |
| **흐름 제어** | `HumanInTheLoopMiddleware` | 조건부 사람 개입 (approve/edit/reject) |
| **흐름 제어** | `ModelCallLimitMiddleware` | 모델 호출 횟수 상한선 |
| **흐름 제어** | `ToolCallLimitMiddleware` | 도구 호출 횟수 상한선 |
| **복원력** | `ModelFallbackMiddleware` | 모델 장애 시 순서대로 다음 모델 시도 |
| **복원력** | `ToolRetryMiddleware` | 도구 실패 시 지수 백오프 재시도 |
| **복원력** | `ModelRetryMiddleware` | 모델 응답 실패 시 재시도 |
| **성능 향상** | `LLMToolSelectorMiddleware` | LLM이 필요한 도구만 골라서 제공 |
| **성능 향상** | `TodoListMiddleware` | `write_todos` 도구 자동 주입 |
| **성능 향상** | `LLMToolEmulatorMiddleware` | 실제 실행 없이 도구 응답 에뮬레이션 |

### 프로덕션 미들웨어 배치 순서 원칙

> 🔑 **핵심 개념**: 프리빌트 미들웨어를 조합할 때는 **빠르고 저렴한 것을 앞에, 느리고 비싼 것을 뒤에** 배치해요. 앞쪽에서 차단된 요청은 뒤쪽 미들웨어를 거치지 않아 비용을 절약해요.

| 순서 | 카테고리 | 이유 |
|------|---------|------|
| 1 | **보안** (PII) | 모델에 민감 정보가 전달되기 전에 제거 |
| 2 | **흐름 제어** (Limit) | 과도한 호출을 사전에 차단 |
| 3 | **복원력** (Retry, Fallback) | 실패 시 자동 복구 |
| 4 | **성능** (ToolSelector) | 도구 선택 최적화 |
| 5 | **대화 관리** (Summary) | 컨텍스트 압축은 마지막에 |

## 환경 설정

In [1]:
# 환경 변수 로드 (.env 파일에서 API 키 등을 읽어와요)
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# LangSmith 추적 설정 (미들웨어 동작을 시각적으로 확인해요)
import os

# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-V1-Prebuilt-Middleware"

# LangSmith 추적이 활성화되었어요.
# print(f"프로젝트: {os.environ['LANGCHAIN_PROJECT']}")

In [3]:
# ---------------------------------------------------
# 기본 모델과 공통 도구 설정
# ---------------------------------------------------
# V1 API: langchain.chat_models, langchain.tools, langchain.agents 사용
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent


# 기본 모델: gpt-4o-mini (비용 효율, 학생 접근성)
# 다른 옵션: "anthropic:claude-sonnet-4-5", "ollama:llama3"
model = init_chat_model("openai:gpt-4o-mini")


# 실습용 공통 도구 정의
@tool
def search_web(query: str) -> str:
    """웹에서 정보를 검색해요."""
    return f"[웹 검색] '{query}'에 대한 결과: 관련 문서 3건 발견"


@tool
def get_stock_price(ticker: str) -> str:
    """주식 현재가를 조회해요."""
    prices = {"AAPL": "$189.30", "GOOGL": "$142.65", "005930": "78,000원"}
    return prices.get(ticker, f"{ticker}: 시세 정보 없음")


@tool
def calculate(expression: str) -> str:
    """수학 표현식을 계산해요."""
    try:
        result = eval(expression)  # 실습용 간단 계산기
        return f"계산 결과: {result}"
    except Exception as e:
        return f"계산 오류: {e}"


# 모델과 도구 설정 완료!
print(f"모델: gpt-4o-mini")
print(f"도구: {search_web.name}, {get_stock_price.name}, {calculate.name}")

모델: gpt-4o-mini
도구: search_web, get_stock_price, calculate


### create_agent 에이전트 그래프 구조

`create_agent()`로 생성한 에이전트는 모두 동일한 기본 그래프 구조를 가져요: `START → agent → tools → agent → ... → END`. 미들웨어는 이 그래프의 agent 노드 **내부**에서 동작해요.

## 1. 대화 관리: SummarizationMiddleware

대화가 길어지면 토큰 비용이 증가하고 최종적으로는 모델의 컨텍스트 창 한도에 도달해요. `SummarizationMiddleware`는 이 문제를 자동으로 해결해요.

> 🔁 **복습 연결**: `01-Middleware-Basics.ipynb` §2에서 `SummarizationMiddleware`의 기본형을 잠깐 봤어요. 여기서는 `trigger`/`keep` 옵션을 깊게 다룹니다.

### 트리거(trigger) 옵션

| 트리거 형식 | 예시 | 설명 |
|------------|------|------|
| `("tokens", N)` | `("tokens", 4000)` | 메시지 총 토큰이 N 초과 시 요약 |
| `("messages", N)` | `("messages", 20)` | 메시지 수가 N 초과 시 요약 |
| `("fraction", F)` | `("fraction", 0.7)` | 모델 컨텍스트 창의 F 비율 초과 시 요약 |

여러 트리거를 리스트로 전달하면 **OR 조건**으로 작동해요 — 어느 하나라도 만족하면 요약이 실행돼요.

### keep 옵션

| keep 형식 | 예시 | 설명 |
|----------|------|------|
| `("messages", N)` | `("messages", 10)` | 요약 후 최근 N개 메시지 유지 |
| `("tokens", N)` | `("tokens", 2000)` | 요약 후 최근 N 토큰에 해당하는 메시지 유지 |

> 💡 **실무 팁**: `trigger=("fraction", 0.7)`을 사용하면 모델마다 다른 컨텍스트 창 크기를 자동으로 처리해요. `gpt-4o-mini`(128K)와 `gpt-4o`(128K) 모두 동일한 설정으로 작동해요. 고정된 토큰 수 대신 비율 기반 트리거가 더 이식성이 높아요.

In [4]:
# ---------------------------------------------------
# SummarizationMiddleware: 대화 자동 요약
# ---------------------------------------------------
# 대화 히스토리가 너무 길어지기 전에 자동으로 요약해요
from langchain.agents.middleware import SummarizationMiddleware

# 방법 1: 토큰 수 기반 트리거 (가장 일반적)
agent_token_trigger = create_agent(
    model=model,
    tools=[search_web],
    middleware=[
        SummarizationMiddleware(
            model="openai:gpt-4o-mini",  # 요약에 사용할 모델 (메인 모델과 달라도 돼요)
            trigger=("tokens", 4000),    # 4000 토큰 초과 시 요약 트리거
            keep=("messages", 10),       # 요약 후 최근 10개 메시지 보존
        ),
    ],
)

# 방법 1: 토큰 기반 트리거
#   trigger=(tokens, 4000): 메시지 총 토큰 > 4000 → 요약 실행
#   keep=(messages, 10): 요약 후 최근 10개 메시지 유지
print()

# 방법 2: 비율 기반 트리거 (모델 독립적)
agent_fraction_trigger = create_agent(
    model=model,
    tools=[search_web],
    middleware=[
        SummarizationMiddleware(
            model="openai:gpt-4o-mini",
            trigger=("fraction", 0.7),   # 컨텍스트 창의 70% 사용 시 요약
            keep=("tokens", 2000),       # 요약 후 최근 2000 토큰 분량 유지
        ),
    ],
)

# 방법 2: 비율 기반 트리거 (이식성 높음)
#   trigger=(fraction, 0.7): 컨텍스트 창 70% 사용 시 요약 실행
#   keep=(tokens, 2000): 요약 후 최근 2000 토큰 분량 유지

In [5]:
# ---------------------------------------------------
# OR 조건: 여러 트리거를 리스트로 전달
# ---------------------------------------------------
# 어느 하나라도 만족하면 요약 실행 (OR 논리)
agent_multi_trigger = create_agent(
    model=model,
    tools=[search_web],
    middleware=[
        SummarizationMiddleware(
            model="openai:gpt-4o-mini",
            trigger=[
                ("tokens", 4000),     # 조건 1: 토큰 4000 초과
                ("messages", 20),     # 조건 2: 메시지 20개 초과
            ],  # OR 조건: 둘 중 하나라도 만족 시 요약
            keep=("messages", 5),
        ),
    ],
)

# 테스트 실행
result = agent_multi_trigger.invoke(
    {"messages": [{"role": "user", "content": "LangGraph 미들웨어에 대해 설명해줘요"}]}
)

# OR 트리거 에이전트 응답:
print(result["messages"][-1].content[:300])

LangGraph 미들웨어에 대한 구체적인 정보는 검색 결과에서 확인할 수 없었습니다. 하지만 일반적으로 미들웨어는 소프트웨어 애플리케이션의 중간에서 데이터를 처리하고 연결하는 역할을 합니다. LangGraph와 같은 특정 미들웨어는 아마도 언어 처리와 관련된 기능을 제공하는 플랫폼일 가능성이 높습니다.

추가적인 정보를 원하시면 LangGraph의 공식 웹사이트나 기술 문서를 참조하시거나, 관련된 소스에서 정보를 찾아보시는 것이 좋습니다. 구체적인 기능이나 사용 사례에 대해 더 알고 싶으시면 질문해 주세요!


## 2. 보안: PIIMiddleware

개인식별정보(PII: Personally Identifiable Information)가 모델에 전달되거나 로그에 기록되는 것을 방지해요. 금융, 의료, 고객 서비스 분야에서는 필수 미들웨어예요.

> 🔁 **복습 연결**: `01` §2에서 `PIIMiddleware`로 맛본 개인정보 보호를, 여기서는 5개 내장 유형 × 4개 전략 옵션으로 확장합니다. 가드레일 계층으로의 배치(Defense in Depth)는 `05-Guardrails.ipynb`에서 다뤄요.

### 지원하는 PII 유형

| PII 유형 | 감지 대상 | 예시 |
|---------|---------|------|
| `"email"` | 이메일 주소 | `user@example.com` |
| `"credit_card"` | 신용카드 번호 | `1234-5678-9012-3456` |
| `"ip"` | IP 주소 | `192.168.1.100` |
| `"mac_address"` | MAC 주소 | `00:1A:2B:3C:4D:5E` |
| `"url"` | URL 주소 | `https://private.example.com` |
| `str` (정규식) | 커스텀 패턴 | `r"010-\d{4}-\d{4}"` |

### 처리 전략 (strategy)

| 전략 | 결과 예시 | 설명 |
|-----|---------|------|
| `"redact"` | `[EMAIL REDACTED]` | 완전히 제거하고 태그로 대체 |
| `"mask"` | `jo***@example.com` | 일부만 표시, 나머지 마스킹 |
| `"hash"` | `[EMAIL:a3f9b2...]` | 해시값으로 대체 (익명화) |
| `"block"` | (예외 발생) | 요청 자체를 차단 |

> 🎯 **강의 포인트**: `apply_to_input`은 사용자 입력을, `apply_to_output`은 모델 응답을, `apply_to_tool_results`는 도구 실행 결과를 처리해요. 기본값은 `apply_to_input=True`, `apply_to_output=False`, `apply_to_tool_results=False`예요. 즉 **기본은 입력만 필터링**하므로, 모델 응답이나 도구 결과의 PII도 막으려면 `apply_to_output=True`를 명시해야 해요. (출처: https://docs.langchain.com/oss/python/langchain/middleware/built-in.md )

> ⚠️ **자주 하는 실수**: `strategy="block"`은 PII가 감지되면 **요청 자체를 차단**해요. 챗봇처럼 사용자 경험이 중요한 곳에서는 `"redact"` 또는 `"mask"`를 사용하는 것이 더 자연스러워요.

In [6]:
# ---------------------------------------------------
# PIIMiddleware: 개인정보 보호 기본 설정
# ---------------------------------------------------
from langchain.agents.middleware import PIIMiddleware

# 여러 PII 유형을 각각 다른 전략으로 처리해요
pii_agent = create_agent(
    model=model,
    tools=[search_web],
    middleware=[
        # 이메일: 완전 제거 (가장 강력한 보호)
        PIIMiddleware("email", strategy="redact"),
        # 신용카드: 마스킹 (뒤 4자리만 표시)
        PIIMiddleware("credit_card", strategy="mask"),
        # IP 주소: 해시로 익명화
        PIIMiddleware("ip", strategy="hash"),
    ],
)

# PII가 포함된 사용자 입력으로 테스트
test_message = (
    "제 이메일은 kim.chulsoo@company.co.kr이에요. "
    "신용카드 번호 4532-1234-5678-9012로 결제하고 싶어요. "
    "서버 IP 10.0.0.254에서 접속 중이에요. "
    "AI 에이전트 보안에 대해 알려줘요!"
)

# === 원본 메시지 ===
print(test_message)
print()

result = pii_agent.invoke(
    {"messages": [{"role": "user", "content": test_message}]}
)

# === PII 처리 후 응답 ===
print(result["messages"][-1].content)

제 이메일은 kim.chulsoo@company.co.kr이에요. 신용카드 번호 4532-1234-5678-9012로 결제하고 싶어요. 서버 IP 10.0.0.254에서 접속 중이에요. AI 에이전트 보안에 대해 알려줘요!



AI 에이전트 보안에 관한 정보는 다음과 같습니다:

1. **데이터 보호**: AI 시스템은 사용자 데이터를 보호하기 위해 암호화 및 접근 제어를 사용합니다. 중요한 정보가 무단 접근으로부터 안전해야 합니다.

2. **사용자 인증**: 신뢰성을 높이기 위해 AI 에이전트는 사용자 인증을 통해 적절한 권한이 부여된 사용자만 접근하도록 합니다.

3. **위협 탐지**: AI는 이상 행동을 분석하여 잠재적인 위협이나 공격을 식별할 수 있습니다. 이를 통해 조기 경고 시스템을 구축할 수 있습니다.

4. **규정 준수**: 데이터 보호 및 프라이버시 법규를 준수하는 것이 필수적입니다. AI 시스템은 해당 법규를 이해하고 준수하도록 설계되어야 합니다.

5. **모델의 안전성**: AI 에이전트는 외부 공격으로부터 안전해야 하며, 모델이 악용되는 것을 방지하기 위한 지속적인 테스트와 업데이트가 필요합니다.

이 외에도 AI 에이전트의 특성에 따라 다양한 보안 조치가 필요합니다.


In [7]:
# ---------------------------------------------------
# PIIMiddleware: 커스텀 정규식으로 한국 개인정보 처리
# ---------------------------------------------------
# 한국 특화 패턴: 휴대폰 번호, 주민등록번호 앞 6자리
pii_korean_agent = create_agent(
    model=model,
    tools=[search_web],
    middleware=[
        # 한국 휴대폰 번호 마스킹 (010-XXXX-XXXX 패턴)
        PIIMiddleware(
            "phone",
            detector=r"010[-\s]?\d{4}[-\s]?\d{4}",  # 정규식으로 패턴 지정
            strategy="mask",
        ),
        # 주민등록번호 앞 6자리 완전 제거 (생년월일 형식)
        PIIMiddleware(
            "korean_id_prefix",
            detector=r"\d{6}[-]\d{7}",  # 주민등록번호 전체 패턴
            strategy="redact",
        ),
        # 표준 이메일도 함께 보호
        PIIMiddleware("email", strategy="redact"),
    ],
)

# 한국 개인정보가 포함된 메시지 테스트
korean_pii_message = (
    "전화번호 010-9876-5432로 연락주세요. "
    "이메일은 test@hankyung.com이에요. "
    "개인정보 보호 정책에 대해 알려줘요."
)

# === 한국 PII 처리 테스트 ===
print("입력:", korean_pii_message[:80])
print()

result = pii_korean_agent.invoke(
    {"messages": [{"role": "user", "content": korean_pii_message}]}
)

print("응답:", result["messages"][-1].content)

입력: 전화번호 010-9876-5432로 연락주세요. 이메일은 test@hankyung.com이에요. 개인정보 보호 정책에 대해 알려줘요.



응답: 개인정보 보호 정책에 대한 정보를 찾았습니다. 일반적으로 개인정보 보호 정책은 다음과 같은 내용을 포함합니다:

1. **개인정보 수집**: 어떤 개인 정보를 수집하는지, 수집 방법 (예: 웹사이트, 앱 등).

2. **이용 목적**: 수집한 개인정보를 어떤 목적으로 사용하는지 (예: 서비스 제공, 마케팅 등).

3. **보관 기간**: 개인정보를 얼마나 오랫동안 보관하는지.

4. **제3자 제공**: 수집한 개인정보를 제3자에게 제공하는 경우, 그 대상과 목적.

5. **안전성**: 개인정보를 보호하기 위해 어떤 안전조치를 취하는지.

6. **사용자의 권리**: 사용자가 자신의 개인정보에 대해 어떤 권리를 가지는지 (예: 열람, 수정, 삭제 요청).

7. **정책 변경**: 개인정보 보호 정책이 변경되는 경우, 어떻게 공지하는지.

구체적인 정책은 각 기업이나 기관에 따라 다를 수 있으므로, 해당 기관의 공식 웹사이트를 방문하여 내용을 확인하는 것이 좋습니다. 추가적인 정보가 필요하시다면 말씀해 주세요!


## 3. 흐름 제어: ModelCallLimitMiddleware, ToolCallLimitMiddleware

에이전트가 무한 루프에 빠지거나 의도치 않게 많은 API 호출을 하는 것을 방지해요. 개발 중 디버깅이나 비용 관리에 유용해요.

> 🔁 **복습 연결**: `01` §2에서 호출 제한 미들웨어를 소개했어요. 여기서는 `thread_limit`/`run_limit`/`exit_behavior` 옵션 차이에 집중합니다.

### thread_limit vs run_limit

| 파라미터 | 범위 | 예시 |
|---------|------|------|
| `thread_limit` | 전체 대화 스레드 누적 | 체크포인터와 함께 사용 |
| `run_limit` | 단일 `.invoke()` 호출 내 | 루프 방지, 비용 제한 |

### exit_behavior 옵션

| 값 | 동작 |
|----|------|
| `"end"` | 현재 응답으로 종료 (기본값) |
| `"error"` | `ModelCallLimitError` 예외 발생 |

> ⚠️ **자주 하는 실수**: `run_limit`을 너무 낮게 설정하면 도구 호출이 많은 작업이 중간에 잘려요. 도구 하나당 최소 2회의 모델 호출(도구 선택 + 결과 해석)이 필요하다는 것을 기억해요.

In [8]:
# ---------------------------------------------------
# ModelCallLimitMiddleware: 모델 호출 횟수 제한
# ---------------------------------------------------
from langchain.agents.middleware import ModelCallLimitMiddleware

limited_agent = create_agent(
    model=model,
    tools=[search_web, calculate],
    middleware=[
        ModelCallLimitMiddleware(
            run_limit=5,          # 단일 invoke 내 최대 5번 모델 호출
            exit_behavior="end",  # 한도 도달 시 종료 (예외 발생 없음)
        ),
    ],
)

# 모델 호출 제한 에이전트 생성 완료
#   run_limit=5: 단일 invoke에서 최대 5번 모델 호출
#   exit_behavior=end: 한도 도달 시 현재 응답으로 종료
print()

result = limited_agent.invoke(
    {"messages": [{"role": "user", "content": "삼성전자 주식 가격 검색하고 1000주 살 때 비용을 계산해줘요"}]}
)

print("응답:", result["messages"][-1].content)

응답: 현재 삼성전자 주식 가격을 정확히 알 수 없지만, 제가 접근할 수 있는 최신 정보로는 검색 결과가 있습니다. 다만 정확한 주식 가격을 제시할 수 있는 최신 정보를 가져오는 것은 불가능합니다.

주가가 예를 들어 75,000원이거나 다른 가격이라고 가정하고 1000주를 구매할 경우, 비용은 다음과 같이 계산할 수 있습니다:

- **주가**: 75,000원
- **주식 수**: 1000주
- **비용**: 75,000 * 1000 = 75,000,000원

정확한 주식 가격에 따라 계산하시면 됩니다. 삼성전자 주가의 현재 상태가 필요하시다면 금융 관련 웹사이트나 앱을 참고해 주세요.


In [9]:
# ---------------------------------------------------
# ToolCallLimitMiddleware: 도구 호출 횟수 제한
# ---------------------------------------------------
from langchain.agents.middleware import ToolCallLimitMiddleware

# 특정 도구(get_stock_price)에만 호출 제한 적용
# 비용이 많이 드는 외부 API 호출을 제한할 때 유용해요
tool_limited_agent = create_agent(
    model=model,
    tools=[search_web, get_stock_price, calculate],
    middleware=[
        # 모든 도구 호출을 합산하여 제한 (전역)
        ToolCallLimitMiddleware(
            run_limit=10,  # 단일 invoke에서 모든 도구 합산 10회 제한
        ),
        # 특정 도구(get_stock_price)에만 별도 제한 추가
        ToolCallLimitMiddleware(
            tool_name="get_stock_price",  # 이 도구에만 적용
            run_limit=3,                  # get_stock_price는 최대 3회
        ),
    ],
)

# 도구 호출 제한 에이전트 생성 완료
#   전역 제한: 모든 도구 합산 최대 10회
#   get_stock_price 전용 제한: 최대 3회
print()

result = tool_limited_agent.invoke(
    {"messages": [{"role": "user", "content": "AAPL, GOOGL 주가를 알려줘요"}]}
)

print("응답:", result["messages"][-1].content)

응답: AAPL의 현재 주가는 $189.30이고, GOOGL의 현재 주가는 $142.65입니다.


## 4. 복원력: ModelFallbackMiddleware, ToolRetryMiddleware

네트워크 오류, API 서버 장애, 일시적인 오류에도 에이전트가 계속 동작하도록 해요.

> 🔁 **복습 연결**: `01` §2에서 `ModelFallbackMiddleware`/`ToolRetryMiddleware`를 맛봤다면, 여기서는 폴백 체인과 지수 백오프 파라미터를 깊게 봅니다.

### ModelFallbackMiddleware

기본 모델이 실패할 때 등록된 순서대로 다음 모델을 시도해요. 캐스케이딩(cascading) 폴백이라고 불러요.

```
gpt-4o-mini (기본) → 실패 → gpt-4o → 실패 → claude-haiku
```

### ToolRetryMiddleware

도구 호출이 실패할 때 지수 백오프(exponential backoff)로 재시도해요.

| 파라미터 | 기본값 | 설명 |
|---------|-------|------|
| `max_retries` | 3 | 최대 재시도 횟수 |
| `initial_delay` | 1.0 | 첫 재시도 전 대기 시간 (초) |
| `backoff_factor` | 2.0 | 대기 시간 배수 (1초 → 2초 → 4초) |
| `max_delay` | 30.0 | 최대 대기 시간 (초) |
| `jitter` | True | 무작위 지터 추가 |

> 💡 **실무 팁**: `jitter=True`는 동시에 많은 요청이 실패했을 때 모두 같은 시간에 재시도하여 서버를 다시 과부하시키는 **Thundering Herd** 문제를 방지해요. 프로덕션에서는 항상 켜두는 것을 권장해요.

In [10]:
# ---------------------------------------------------
# ModelFallbackMiddleware: 캐스케이딩 모델 폴백
# ---------------------------------------------------
from langchain.agents.middleware import ModelFallbackMiddleware

# 기본 모델 실패 시 순서대로 다음 모델 시도
fallback_agent = create_agent(
    model="openai:gpt-4o-mini",  # 기본 모델 (1순위)
    tools=[search_web],
    middleware=[
        ModelFallbackMiddleware(
            "openai:gpt-4o",                   # 2순위: gpt-4o
            "anthropic:claude-haiku-4-5",      # 3순위: claude-haiku
        ),
    ],
)

# 모델 폴백 에이전트 생성 완료
#   1순위: gpt-4o-mini (기본)
#   2순위: gpt-4o (폴백 1)
#   3순위: claude-haiku-4-5 (폴백 2)
print()

result = fallback_agent.invoke(
    {"messages": [{"role": "user", "content": "AI 에이전트의 폴백 전략이 왜 중요한지 설명해줘요"}]}
)

print("응답:", result["messages"][-1].content[:300])

응답: AI 에이전트의 폴백 전략은 여러 면에서 중요합니다:

1. **신뢰성 확보**: AI 에이전트가 예상치 못한 상황이나 오류 발생 시 폴백 전략을 통해 안정적으로 작동할 수 있습니다. 이는 사용자에게 신뢰를 제공하며 시스템의 신뢰성을 높입니다.

2. **위험 관리**: 폴백 전략은 시스템이 실패하거나 오류가 발생했을 때의 대처 방안을 마련하여 잠재적인 위험을 최소화합니다. 이를 통해 시스템이 크리티컬한 상황에서도 안전하게 운영될 수 있습니다.

3. **지속성 유지**: 중요한 작업이나 서비스를 중단 없이 지속하기 위해 폴백 전략


In [11]:
# ---------------------------------------------------
# ToolRetryMiddleware: 지수 백오프 재시도
# ---------------------------------------------------
from langchain.agents.middleware import ToolRetryMiddleware

# 외부 API 호출 실패 시 자동으로 재시도해요
retry_agent = create_agent(
    model=model,
    tools=[search_web, get_stock_price],
    middleware=[
        ToolRetryMiddleware(
            max_retries=3,       # 최대 3번 재시도
            initial_delay=1.0,   # 첫 재시도 전 1초 대기
            backoff_factor=2.0,  # 대기 시간 2배씩 증가: 1초 → 2초 → 4초
            max_delay=30.0,      # 최대 30초까지만 대기 (이후 포기)
            jitter=True,         # 무작위 지터 추가 (Thundering Herd 방지)
        ),
    ],
)

# 도구 재시도 에이전트 생성 완료
#   재시도 간격: 1초 → 2초 → 4초 (+ 무작위 지터)
#   최대 대기: 30초
print()

result = retry_agent.invoke(
    {"messages": [{"role": "user", "content": "AAPL 주가를 검색해서 알려줘요"}]}
)

print("응답:", result["messages"][-1].content)

응답: AAPL의 현재 주가는 $189.30입니다.


## 5. 성능 향상: LLMToolSelectorMiddleware

도구가 많을수록 모델의 컨텍스트를 소비하고, LLM이 올바른 도구를 선택하기 어려워져요. `LLMToolSelectorMiddleware`는 현재 대화에 실제로 필요한 도구만 선별하여 모델에 제공해요.

이 미들웨어는 내부적으로 **별도의 LLM 호출**로 도구를 선별해요:
1. 현재 대화 내용을 분석해요
2. 전체 도구 목록 중 필요한 것만 골라요
3. 선별된 도구만 메인 모델에 전달해요

> 🎯 **강의 포인트**: 도구가 5개 이하일 때는 오히려 불필요한 LLM 호출이 추가되어 비효율적이에요. **10개 이상의 도구**를 가진 에이전트에서 효과가 두드러져요. 도구 수와 `max_tools` 설정에 따른 성능 트레이드오프를 이해하는 것이 중요해요.

In [12]:
# ---------------------------------------------------
# LLMToolSelectorMiddleware: 필요한 도구만 선택
# ---------------------------------------------------
from langchain.agents.middleware import LLMToolSelectorMiddleware
from langchain.tools import tool


# 많은 도구를 가진 시나리오 시뮬레이션
@tool
def send_email(to: str, subject: str, body: str) -> str:
    """이메일을 전송해요."""
    return f"이메일 전송 완료: {to}에게 '{subject}'"


@tool
def create_calendar_event(title: str, date: str, time: str) -> str:
    """캘린더에 이벤트를 추가해요."""
    return f"이벤트 생성: {title} ({date} {time})"


@tool
def query_database(sql: str) -> str:
    """데이터베이스를 쿼리해요."""
    return f"쿼리 결과: {sql[:50]}... → 15개 행 반환"


@tool
def translate_text(text: str, target_lang: str) -> str:
    """텍스트를 번역해요."""
    return f"번역 결과 ({target_lang}): {text[:50]}..."


# 많은 도구를 보유한 에이전트 (도구 선택기가 효과적)
all_tools = [
    search_web,
    get_stock_price,
    calculate,
    send_email,
    create_calendar_event,
    query_database,
    translate_text,
]

tool_selector_agent = create_agent(
    model=model,
    tools=all_tools,  # 7개 도구 보유
    middleware=[
        LLMToolSelectorMiddleware(
            model="openai:gpt-4o-mini",  # 도구 선별용 모델 (가볍게 선택)
            max_tools=3,                 # 매 호출마다 최대 3개 도구만 전달
            always_include=["search_web"],  # 항상 포함할 도구 지정
        ),
    ],
)

print(f"전체 도구 수: {len(all_tools)}개")
#   LLMToolSelectorMiddleware: 매 호출 시 최대 3개만 선별
#   search_web은 항상 포함
print()

# 주식 관련 질문 → get_stock_price, calculate 선별 예상
result = tool_selector_agent.invoke(
    {"messages": [{"role": "user", "content": "AAPL 주가로 100만원어치 살 수 있는 주식 수를 계산해줘요"}]}
)

print("응답:", result["messages"][-1].content)

전체 도구 수: 7개



응답: 현재 AAPL 주가는 $189.30입니다. 100만원어치의 AAPL 주식을 살 수 있는 주식 수는 약 5282주입니다.


## 6. 성능 향상: TodoListMiddleware

`TodoListMiddleware`는 에이전트에 `write_todos` 도구를 자동으로 주입해요. 이 도구를 통해 에이전트가 복잡한 작업을 처리할 때 할 일 목록을 관리하고, 진행 상황을 추적할 수 있어요.

에이전트가 `write_todos` 도구를 호출하면:
1. State의 `todos` 필드가 업데이트돼요
2. 사용자에게 진행 상황이 명확하게 보여요
3. 복잡한 멀티스텝 작업의 완료율을 추적할 수 있어요

> 🔑 **핵심 개념**: `TodoListMiddleware`는 도구를 **주입(inject)**하는 미들웨어예요. `create_agent`의 `tools` 파라미터에 직접 추가하지 않아도, 미들웨어가 에이전트 실행 중 자동으로 도구를 추가해요.

In [13]:
# ---------------------------------------------------
# TodoListMiddleware: write_todos 도구 자동 주입
# ---------------------------------------------------
from langchain.agents.middleware import TodoListMiddleware

# write_todos 도구를 자동으로 에이전트에 추가해요
todo_agent = create_agent(
    model=model,
    tools=[search_web, calculate],  # 기본 도구만 정의 (write_todos는 자동 주입)
    middleware=[
        TodoListMiddleware(),  # write_todos 도구를 에이전트에 자동 주입
    ],
)

# 복잡한 멀티스텝 작업: 에이전트가 할 일 목록을 만들며 진행
result = todo_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "AAPL 주가 조사하고, GOOGL 주가도 조사한 후, "
                    "두 주식을 100만원씩 살 때 총 비용을 계산해줘요. "
                    "각 단계를 할 일 목록으로 관리해줘요."
                ),
            }
        ]
    }
)

# === 최종 응답 ===
print(result["messages"][-1].content)

# State의 todos 확인
if "todos" in result:
    print()
    # === 할 일 목록 (State.todos) ===
    for todo in result["todos"]:
        status = "✅" if todo.get("done") else "⬜"
        print(f"  {status} {todo.get('task', todo)}")

AAPL과 GOOGL의 주가 조사를 완료했습니다. 이제 두 주식을 100만원씩 샀을 때의 총 비용을 계산할 준비가 되었습니다.

AAPL 주가와 GOOGL 주가에 대한 정보를 사용하여 100만원씩 구입할 경우의 총 비용을 계산해보겠습니다. 주가는 조사한 결과에 따라 다를 수 있으니, 추가적인 정보가 필요합니다. 실제 주가를 알려주시면 계산해드리겠습니다.

  ⬜ {'content': 'AAPL 주가 조사하기', 'status': 'completed'}
  ⬜ {'content': 'GOOGL 주가 조사하기', 'status': 'completed'}
  ⬜ {'content': 'AAPL과 GOOGL 100만원씩 매수할 때 총 비용 계산하기', 'status': 'pending'}


## 7. 대화 관리: ContextEditingMiddleware

`ContextEditingMiddleware`는 대화 히스토리에서 특정 유형의 메시지를 제거해요. 가장 일반적인 용도는 `ClearToolUsesEdit`로 도구 사용 기록(ToolMessage)을 제거하는 것이에요.

**도구 사용 기록을 제거하는 이유:**
- 반복적인 도구 호출 결과가 누적되면 컨텍스트를 낭비해요
- 특정 시점 이후로는 과거 도구 결과가 불필요해요
- 요약 전 불필요한 도구 기록을 제거하여 요약 품질을 높여요

> 💡 **실무 팁**: `ContextEditingMiddleware`와 `SummarizationMiddleware`를 함께 쓸 때는 `ContextEditingMiddleware`를 **먼저** 배치하세요. 도구 기록을 제거한 후 요약하면 요약 내용이 더 정제되어요.

In [ ]:
# ---------------------------------------------------
# ContextEditingMiddleware: 도구 사용 기록 정리
# ---------------------------------------------------
from langchain.agents.middleware import (
    ContextEditingMiddleware,
    ClearToolUsesEdit,
)

# 도구 사용 기록이 N개를 초과하면 오래된 것부터 제거해요
# ClearToolUsesEdit 파라미터:
#   trigger: 총 토큰 수가 이 값을 초과하면 정리 시작 (기본값: 100000)
#   keep: 정리 후 유지할 도구 사용 기록 수 (기본값: 3)
context_editing_agent = create_agent(
    model=model,
    tools=[search_web, get_stock_price, calculate],
    middleware=[
        ContextEditingMiddleware(
            edits=[
                ClearToolUsesEdit(
                    trigger=50000,  # 50000 토큰 초과 시 정리 시작
                    keep=5,         # 도구 사용 기록 최대 5개 유지
                ),
            ]
        ),
        # 도구 기록 정리 후 요약 (순서 중요!)
        SummarizationMiddleware(
            model="openai:gpt-4o-mini",
            trigger=("tokens", 3000),
            keep=("messages", 8),
        ),
    ],
)

# 컨텍스트 편집 + 요약 에이전트 생성 완료
#   순서 1: ContextEditingMiddleware - 50000 토큰 초과 시 도구 기록 정리, 최대 5개 유지
#   순서 2: SummarizationMiddleware - 3000 토큰 초과 시 요약
print()

result = context_editing_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "AAPL 주가를 검색하고 100주 살 때 비용을 계산해줘요"}
        ]
    }
)

print("응답:", result["messages"][-1].content)

응답: AAPL의 현재 주가는 $189.30이며, 100주를 살 경우 비용은 $18,930.00입니다.


## 8. 종합 예제: 프로덕션 레벨 에이전트

실무에서는 여러 프리빌트 미들웨어를 조합하여 안정적이고 안전한 에이전트를 만들어요. 이 예제는 다음 미들웨어를 함께 적용해요:

| 순서 | 미들웨어 | 역할 |
|-----|---------|------|
| 1 | `PIIMiddleware` | 사용자 입력의 개인정보 보호 (가장 먼저!) |
| 2 | `ModelCallLimitMiddleware` | 비용 과다 방지 |
| 3 | `ToolRetryMiddleware` | 외부 API 실패 복원 |
| 4 | `ModelFallbackMiddleware` | 모델 장애 대비 |
| 5 | `LLMToolSelectorMiddleware` | 도구 선택 최적화 |
| 6 | `ContextEditingMiddleware` | 도구 기록 정리 |
| 7 | `SummarizationMiddleware` | 컨텍스트 압축 |

> 🎯 **강의 포인트**: 미들웨어 순서는 에이전트의 **동작 방식**을 결정해요. PII 보호는 항상 앞에, 컨텍스트 관리(요약)는 항상 마지막에 배치하는 것이 일반적인 원칙이에요.

In [15]:
# ---------------------------------------------------
# 프로덕션 레벨 에이전트: 7개 미들웨어 조합
# ---------------------------------------------------
production_agent = create_agent(
    model="openai:gpt-4o-mini",  # 기본 모델
    tools=[search_web, get_stock_price, calculate],
    middleware=[
        # 1. 보안: 개인정보 보호 (항상 최우선)
        PIIMiddleware("email", strategy="redact"),
        PIIMiddleware("credit_card", strategy="mask"),

        # 2. 비용 제어: 호출 횟수 제한
        ModelCallLimitMiddleware(
            run_limit=8,
            exit_behavior="end",
        ),
        ToolCallLimitMiddleware(run_limit=6),

        # 3. 복원력: 실패 시 자동 복구
        ToolRetryMiddleware(
            max_retries=2,
            initial_delay=0.5,
            backoff_factor=2.0,
            jitter=True,
        ),
        ModelFallbackMiddleware("openai:gpt-4o"),  # 폴백 모델 1개

        # 4. 성능: 도구 선택 최적화
        LLMToolSelectorMiddleware(
            model="openai:gpt-4o-mini",
            max_tools=2,
        ),

        # 5. 메모리 관리: 컨텍스트 압축 (항상 마지막)
        # ClearToolUsesEdit: trigger=토큰 수 초과 시 정리, keep=유지할 기록 수
        ContextEditingMiddleware(edits=[ClearToolUsesEdit(trigger=50000, keep=4)]),
        SummarizationMiddleware(
            model="openai:gpt-4o-mini",
            trigger=[("tokens", 3000), ("messages", 15)],
            keep=("messages", 8),
        ),
    ],
)

# 프로덕션 레벨 에이전트 생성 완료
# 적용된 미들웨어:
#   1. PIIMiddleware (email, credit_card)
#   2. ModelCallLimitMiddleware (run_limit=8)
#   3. ToolCallLimitMiddleware (run_limit=6)
#   4. ToolRetryMiddleware (max_retries=2)
#   5. ModelFallbackMiddleware (gpt-4o)
#   6. LLMToolSelectorMiddleware (max_tools=2)
#   7. ContextEditingMiddleware
#   8. SummarizationMiddleware

In [ ]:
# 그래프 흐름: START → agent → tools → agent → ... → END
# agent 노드: 7개 미들웨어가 순서대로 적용돼요
# PII 보호 → 호출 제한 → 도구 재시도 → 모델 폴백 → 도구 선택 → 컨텍스트 편집 → 요약
# tools 노드: 도구 호출 실행 (재시도, 호출 제한 미들웨어가 보호)
from IPython.display import Image, display

display(Image(production_agent.get_graph().draw_mermaid_png()))

In [17]:
# ---------------------------------------------------
# 프로덕션 에이전트 테스트 실행
# ---------------------------------------------------
# PII가 포함되고 도구가 필요한 복합 요청
complex_request = (
    "저는 kim@company.com이에요. "
    "AAPL 주가를 검색하고 500만원어치 살 때 주식 수를 계산해줘요. "
    "결과를 간결하게 요약해서 알려줘요."
)

# === 입력 메시지 ===
print(complex_request)
print()

result = production_agent.invoke(
    {"messages": [{"role": "user", "content": complex_request}]}
)

# === 에이전트 응답 ===
print(result["messages"][-1].content)

저는 kim@company.com이에요. AAPL 주가를 검색하고 500만원어치 살 때 주식 수를 계산해줘요. 결과를 간결하게 요약해서 알려줘요.



AAPL 주가는 $189.30입니다. 500만원어치 주식을 살 경우 약 26,413주를 구매할 수 있습니다.


## 9. 실습: 맞춤형 미들웨어 스택 설계

아래 시나리오에 맞는 미들웨어 스택을 설계하고 구현해봐요.

In [18]:
# ============================================================
# 실습 해설: 의료 상담 챗봇을 위한 미들웨어 스택 설계
#
# 요구사항:
#   1. 사용자 입력의 이메일과 전화번호를 보호해야 해요
#   2. 의료 상담 기록이 길어지면 자동으로 요약해야 해요
#   3. 외부 의약품 데이터베이스 API 호출이 실패하면 재시도해요
#   4. 단일 상담 세션에서 모델 호출이 10회를 넘으면 안 돼요
#
# 순서: PII 보호 → 호출 제한 → 재시도 → 컨텍스트 관리
# ============================================================


@tool
def drug_database_lookup(drug_name: str) -> str:
    """의약품 정보를 데이터베이스에서 조회해요."""
    drugs = {
        "타이레놀": "성분: 아세트아미노펜 500mg. 효능: 해열·진통. 주의: 간 질환자 주의.",
        "아스피린": "성분: 아스피린 100mg. 효능: 혈전 예방. 주의: 출혈 위험.",
    }
    return drugs.get(drug_name, f"{drug_name}: 데이터베이스에 없는 약품이에요")


@tool
def symptom_checker(symptoms: str) -> str:
    """증상을 기반으로 가능한 질환을 안내해요."""
    return f"증상 '{symptoms}' 분석 결과: 의사 상담이 권장됩니다. (면책 조항: 의학적 조언이 아닙니다)"


medical_chatbot = create_agent(
    model=model,
    tools=[drug_database_lookup, symptom_checker],
    middleware=[
        PIIMiddleware("email", strategy="redact"),
        PIIMiddleware(
            "phone",
            detector=r"01[016789][-\s]?\d{3,4}[-\s]?\d{4}",
            strategy="mask",
        ),
        ModelCallLimitMiddleware(run_limit=10, exit_behavior="end"),
        ToolRetryMiddleware(
            max_retries=2,
            initial_delay=0.5,
            backoff_factor=2.0,
            max_delay=5.0,
            jitter=True,
        ),
        SummarizationMiddleware(
            model="openai:gpt-4o-mini",
            trigger=[("tokens", 2500), ("messages", 12)],
            keep=("messages", 6),
        ),
    ],
    system_prompt=(
        "당신은 의료 정보를 제공하는 챗봇이에요. "
        "진단을 확정하지 말고 일반 정보와 주의사항만 제공하세요. "
        "모든 답변 마지막에 '전문 의료인과 상담하세요'를 덧붙여요."
    ),
)

test_message = (
    "제 이메일 patient@hospital.com이고 전화번호는 010-1234-5678이에요. "
    "두통이 심한데 타이레놀을 먹어도 될까요?"
)

result = medical_chatbot.invoke(
    {"messages": [{"role": "user", "content": test_message}]}
)

print(result["messages"][-1].content)

타이레놀(아세트아미노펜)은 해열 및 진통제로, 두통을 완화하는 데 효과적입니다. 하지만 간 질환이 있는 경우 주의가 필요합니다. 증상이 계속되거나 심해지면 전문 의료인과 상담하세요.


## 10. 실습 파일 안내: cost_tracker.py, language_router.py

이 노트북과 함께 제공되는 두 개의 실습 파일에서 프리빌트 미들웨어를 활용한 실용적인 구현 예시를 확인할 수 있어요.

### `middleware/cost_tracker.py`

`ToolCallLimitMiddleware`와 커스텀 클래스를 조합하여 **토큰 사용량과 API 호출 비용을 실시간으로 추적**해요.

```python
from middleware.cost_tracker import CostTrackerMiddleware

tracker = CostTrackerMiddleware(budget_usd=1.0)  # $1.00 예산 설정
agent = create_agent(model=model, tools=[...], middleware=[tracker])
# 실행 후:
tracker.print_cost_report()  # 비용 보고서 출력
```

### `middleware/language_router.py`

`@dynamic_prompt`를 활용하여 **입력 언어를 자동 감지하고 언어별 시스템 프롬프트로 전환**해요.

```python
from middleware.language_router import LanguageRouterMiddleware

agent = create_agent(
    model=model,
    tools=[...],
    middleware=[LanguageRouterMiddleware()],
)
# 한국어 입력 → 한국어 시스템 프롬프트
# 영어 입력 → 영어 시스템 프롬프트
```

In [19]:
# ---------------------------------------------------
# 실습 파일 사용 예시
# ---------------------------------------------------
# middleware/ 디렉토리의 실습 파일을 불러와서 사용해요
from middleware.cost_tracker import CostTrackerMiddleware
from middleware.language_router import LanguageRouterMiddleware

# 실습 파일 로드 성공!
#   - CostTrackerMiddleware: 토큰 비용 추적
#   - LanguageRouterMiddleware: 언어 자동 감지

# CostTrackerMiddleware 사용 예시
tracker = CostTrackerMiddleware(budget_usd=0.50)  # $0.50 예산

cost_agent = create_agent(
    model=model,
    tools=[search_web],
    middleware=[tracker],
)

result = cost_agent.invoke(
    {"messages": [{"role": "user", "content": "AI 에이전트란 무엇인가요?"}]}
)

print()
print("에이전트 응답:", result["messages"][-1].content[:100], "...")
print()
tracker.print_cost_report()


[CostTracker] 호출 #1 | 입력: 3t, 출력: 0t | 이번 비용: $0.00000 | 누적: $0.00000 | 잔여 예산: $0.5000


[CostTracker] 호출 #2 | 입력: 13t, 출력: 0t | 이번 비용: $0.00000 | 누적: $0.00000 | 잔여 예산: $0.5000


[CostTracker] 호출 #3 | 입력: 25t, 출력: 0t | 이번 비용: $0.00000 | 누적: $0.00001 | 잔여 예산: $0.5000


[CostTracker] 호출 #4 | 입력: 47t, 출력: 0t | 이번 비용: $0.00001 | 누적: $0.00001 | 잔여 예산: $0.5000

에이전트 응답: "AI 에이전트" 또는 "AI agent"는 특정한 작업이나 목표를 달성하기 위해 환경과 상호작용하는 소프트웨어 프로그램이나 시스템을 의미합니다. 이러한 에이전트는 데이터를 수집하 ...


비용 추적 보고서 (Cost Tracker Report)
모델:             gpt-4o-mini
총 호출 횟수:     4회
입력 토큰:        88개
출력 토큰:        0개
총 토큰:          88개
-------------------------------------------------------
총 비용:          $0.00001
설정 예산:        $0.50000
예산 사용률:      0.0%
잔여 예산:        $0.49999
-------------------------------------------------------
호출별 비용 내역:
  #01: 입력     3t, 출력     0t, 비용 $0.00000
  #02: 입력    13t, 출력     0t, 비용 $0.00000
  #03: 입력    25t, 출력     0t, 비용 $0.00000
  #04: 입력    47t, 출력     0t, 비용 $0.00001


In [20]:
# ---------------------------------------------------
# LanguageRouterMiddleware 사용 예시
# ---------------------------------------------------
from middleware.language_router import LanguageRouterMiddleware

lang_agent = create_agent(
    model=model,
    tools=[search_web],
    middleware=[LanguageRouterMiddleware()],
)

# 한국어 입력 테스트
# === 한국어 입력 ===
result_ko = lang_agent.invoke(
    {"messages": [{"role": "user", "content": "LangGraph가 뭔가요?"}]}
)
print(result_ko["messages"][-1].content[:200])
print()

# 영어 입력 테스트
# === English Input ===
result_en = lang_agent.invoke(
    {"messages": [{"role": "user", "content": "What is LangGraph?"}]}
)
print(result_en["messages"][-1].content[:200])


LangGraph는 자연어 처리(NLP) 및 최적화 관련 기술을 활용한 그래프 기반의 데이터 구조입니다. 이러한 그래프는 언어 정보를 시각화하거나 비교하는 데 사용되며, 언어 간의 관계를 탐색할 수 있도록 돕습니다. 기본적으로 언어 모델을 구축하고, 서로 다른 언어 간의 의미적 연결고리를 분석하는 데 활용될 수 있습니다. 

보다 구체적인 정보가 필요하시다



LangGraph appears to be a concept or tool related to language processing, particularly in the fields of AI and natural language understanding. However, detailed specific information is limited in the 


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **SummarizationMiddleware**: `trigger` 옵션으로 tokens/messages/fraction 기반 요약을 트리거하고, 여러 조건을 리스트로 전달하면 OR 논리로 작동해요
- **PIIMiddleware**: email, credit_card, ip, mac_address, url과 커스텀 정규식 패턴을 지원하며, redact/mask/hash/block 4가지 전략으로 처리해요
- **ModelCallLimitMiddleware / ToolCallLimitMiddleware**: `run_limit`으로 단일 실행 내 호출 횟수를 제한하고, `exit_behavior`로 종료 방식을 선택해요
- **ModelFallbackMiddleware**: 기본 모델 실패 시 등록된 순서대로 다음 모델을 캐스케이딩 시도해요
- **ToolRetryMiddleware**: `backoff_factor`로 지수 백오프를, `jitter=True`로 Thundering Herd 방지를 설정해요
- **LLMToolSelectorMiddleware**: `max_tools`와 `always_include`로 각 호출마다 필요한 도구만 선별하여 컨텍스트를 효율화해요
- **TodoListMiddleware**: `write_todos` 도구를 자동 주입하여 복잡한 멀티스텝 작업의 진행 상황을 추적해요
- **ContextEditingMiddleware**: `ClearToolUsesEdit`로 도구 사용 기록이 누적되는 것을 방지해요
- **미들웨어 순서 원칙**: PII 보호 → 호출 제한 → 복원력 → 성능 최적화 → 컨텍스트 관리 순으로 배치해요

## 다음 노트북 예고

다음 `05-Guardrails.ipynb`에서는 **가드레일(Guardrails)**을 배워요. 오늘 배운 `PIIMiddleware`를 넘어 더 강력한 PII 보호(Redact/Mask/Hash/Block)와 커스텀 안전 필터를 직접 구현해볼게요. 특정 주제나 유해 콘텐츠를 완전히 차단하는 프로덕션 수준의 가드레일 시스템을 만들어봐요.